In [3]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [4]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [5]:
len(documents)

72

## QUERTION 2

### Q2. Indexing and searching
### Index the documents with minsearch - make content a text field and filename a keyword field. Then search with this query:

### How does the agentic loop keep calling the model until it stops?

In [6]:
from openai import OpenAI
import os

openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [7]:
def llm(prompt):
    response = openai_client.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

In [8]:
question = "How does the agentic loop keep calling the model until it stops?"
answer = llm(question)
print(answer)

The "agentic loop" you're referring to is often associated with model-based Reinforcement Learning (RL) and Large Language Models (LLMs) like those developed by Meta AI. However, the concept of an "agentic loop" can be applied in a more general sense, often referred to in terms of an "inference loop".

An inference loop is a cycle where a model is used to predict its next action or decision based on its current state. In this cycle, the model:

1. Observes its environment or receives input.
2. Makes a prediction or decision based on its internal state and the input.
3. Takes an action in the environment.
4. Observes the consequences of its action (e.g., the new state of the environment).
5. Updates its internal state based on the observation.

In the context of an LLM, this loop continues until the model is stopped or a terminal condition is reached, such as a timeout or a specific exit command.

The model does not actively "call" itself, but rather, the programming framework or the mo

In [9]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [10]:
results = index.search("How does the agentic loop keep calling the model until it stops?")
print(results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


In [11]:
#question = "I just discovered the course. Can I join now?"



In [12]:
def search(question, course="llm-zoomcamp"):
    return index.search(
        question,
     #   filter_dict={"course": "llm-zoomcamp"},
        num_results=1
    )


In [13]:
search_results = search(question)
print(search_results[0]["filename"])

01-agentic-rag/lessons/14-agentic-loop.md


## Q3. RAG
### Now we will build a RAG assistant on top of this data. Let's use the rag helper script we prepared during the lessons:

In [14]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py


--2026-06-23 03:04:15--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.110.133, 185.199.108.133, 185.199.109.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.110.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py.1’

rag_helper.py.1     100%[===================>]   2.08K  --.-KB/s    in 0.001s  

2026-06-23 03:04:16 (2.11 MB/s) - ‘rag_helper.py.1’ saved [2134/2134]



### Two solutions are possible:

#### Implement the RAG flow yourself
#### Take RAGBase and change the parts related to the FAQ schema - search (to use our index) and build_context
#### Build a RAG over the index from Q2 and answer the query:

### How does the agentic loop keep calling the model until it stops?

In [15]:
from rag_helper import RAGBase
import importlib
import rag_helper
importlib.reload(rag_helper)
#from rag_helper import RAGBase


<module 'rag_helper' from '/app/01-homework/rag_helper.py'>

In [16]:
assistant = RAGBase( index,openai_client)

In [17]:
assistant.rag('How does the agentic loop keep calling the model until it stops?')

'I\'ll answer your questions based on the provided context.\n\nGo ahead and ask your question. I\'ll respond with the relevant information from the context or say "I don\'t know" if the answer is not found in the provided text.'

In [18]:
answer = assistant.rag('How does the agentic loop keep calling the model until it stops?')

assistant.last_usage



CompletionUsage(completion_tokens=17, prompt_tokens=7221, total_tokens=7238, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.242447672, prompt_time=0.696230255, completion_time=0.082124075, total_time=0.77835433)

In [19]:
assistant.last_usage.prompt_tokens


7221

In [20]:
assistant.last_usage.completion_tokens


17

In [21]:
assistant.last_usage.total_tokens

7238

In [22]:
# Q4. Chunking


In [23]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [30]:
len(chunks[0])

3

In [31]:
# Build a new index on the chunks
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)
chunk_index.fit(chunks)

# Search with the same query
results = chunk_index.search(
    "How does the agentic loop keep calling the model until it stops?"
)

print(results[0]["filename"])
print(results[0]["start"])   # offset where this chunk begins in the original page

01-agentic-rag/lessons/14-agentic-loop.md
4000


# Q4. Chunking

In [37]:
assistant_chunk = RAGBase( chunk_index,openai_client)
assistant_chunk
assistant_chunk.rag('How does the agentic loop keep calling the model until it stops?')

'The agentic loop keeps calling the model until it stops by using a `while True` loop that continues to execute until a specific condition is met. The condition is that the model returns a response without any function calls, indicated by the `has_function_calls` flag being `False`.\n\nIn the provided code, the loop iterates as follows:\n\n1. It sends a message to the model using `openai_client.responses.create`.\n2. It checks the response for function calls (`item.type == "function_call"`).\n3. If a function call is found, it executes the call using `make_call(item)` and appends the output to the `messages` list.\n4. It sets `has_function_calls` to `True` if a function call is found.\n5. It increments the iteration counter `it`.\n6. It checks if `has_function_calls` is `False`. If it is, the loop breaks, and the function returns the final answer.\n\nBy continuously calling the model and executing function calls until no more function calls are returned, the agentic loop allows the mod

In [38]:
assistant_chunk.last_usage

CompletionUsage(completion_tokens=240, prompt_tokens=2348, total_tokens=2588, completion_tokens_details=None, prompt_tokens_details=None, queue_time=0.242638671, prompt_time=0.121709427, completion_time=0.695323723, total_time=0.81703315)

In [40]:
assistant_chunk.last_usage.total_tokens

2588

In [42]:
assistant.last_usage.total_tokens/assistant_chunk.last_usage.total_tokens 

2.7967542503863987